In [62]:
import torch
import os 
import json
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR100
from torchvision import transforms, models
%cd /home/devops/data/DL4CV/DL4CV/A1/Retrain
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# TIỀN XỬ LÝ DỮ LIỆU & AUGMENTATION
# ==========================================
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        (0.5071, 0.4867, 0.4408),
        (0.2675, 0.2565, 0.2761)
    ),

    transforms.RandomErasing(
        p=0.25,
        scale=(0.02, 0.15),
        ratio=(0.3, 3.3)
    )
])
# Chuẩn hóa theo Mean và Std của bộ dữ liệu CIFAR-100
# transform_train = transforms.Compose([
#     #transforms.Resize(224),  
#     transforms.RandomCrop(32, padding=4),
#     transforms.RandomHorizontalFlip(),
#     transforms.ToTensor(),
#     transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
# ])

transform_test = transforms.Compose([
    #transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])
# 4. Khởi tạo Dataset chuẩn
raw_train_dataset  = CIFAR100(root="data", train=True, download=True, transform=transform_train)
raw_val_dataset    = CIFAR100(root="data", train=True, download=False, transform=transform_test)
final_test_dataset = CIFAR100(root="data", train=False, download=True, transform=transform_test)

# 5. Chia Train (90%) - Val (10%) từ 50,000 mẫu
total_samples = len(raw_train_dataset)
train_size = int(0.9 * total_samples)
val_size = total_samples - train_size

generator = torch.Generator().manual_seed(42)
permuted_indices = torch.randperm(total_samples, generator=generator).tolist()

train_indices = permuted_indices[:train_size]
val_indices   = permuted_indices[train_size:]

TRAIN_SUBSET = Subset(raw_train_dataset, train_indices)
VAL_SUBSET   = Subset(raw_val_dataset, val_indices)

# 6. DataLoader với num_workers=0 để tránh lỗi Shared Memory (shm)
BATCH_SIZE = 64
train_loader = DataLoader(TRAIN_SUBSET, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(VAL_SUBSET, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(final_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

/home/devops/data/DL4CV/DL4CV/A1/Retrain


In [63]:
def build_resnet18_scratch(num_classes=100):

    # 1. Initialize ResNet-18 WITHOUT pretrained weights
    model = models.resnet18(weights=None)

    # 2. Modify the first convolution for CIFAR-100 (32x32)
    model.conv1 = nn.Conv2d(
        in_channels=3,
        out_channels=64,
        kernel_size=3,
        stride=1,
        padding=1,
        bias=False
    )

    # 3. Remove MaxPooling
    model.maxpool = nn.Identity()

    # 4. Replace the classification layer
    model.fc = nn.Linear(
        in_features=512,
        out_features=num_classes
    )

    return model


In [64]:
# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = build_resnet18_scratch(num_classes=100)
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), p

In [65]:
from count_parameters import count_parameters
params = count_parameters(model)
print(f"Total parameters: {params['total_params']} M")
print(f"Trainable parameters: {params['trainable_params']} M")

Total parameters: 11.220132 M
Trainable parameters: 11.220132 M


In [66]:
from torch.optim.lr_scheduler import StepLR
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.01
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50,
    eta_min=1e-6
)

In [67]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
            
    return running_loss / total, correct / total


In [68]:
import time

from experiment_logger import ExperimentLogger
from plot_training_results import plot_training_results


# ==========================================
# EXPERIMENT CONFIGURATION
# ==========================================

NUM_EPOCHS = 50
SEED = 42
BATCH_SIZE = 64

best_val_acc = 0.0

CHECKPOINT_PATH = "best_resnet18_cifar100.pth"
HISTORY_PATH = "resnet18_cifar100_history_from_scratch.json"

current_run_history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

config = {
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "optimizer": "SGD",
    "learning_rate": optimizer.param_groups[0]["lr"],
    "scheduler": "CosineAnnealingLR",
    "loss_function": "CrossEntropyLoss",
    "pretrained": False,
    "frozen_backbone": False,
    "precision": "FP32"
}


# ==========================================
# INITIALIZE EXPERIMENT LOGGER
# ==========================================

logger = ExperimentLogger(
    experiment_id=f"ResNet18_Scratch_seed{SEED}",
    model_name="ResNet18",
    strategy="Scratch",
    seed=SEED,
    input_size=[32, 32],
    config=config,
    filename="results/experiment_results.json"
)

logger.log_model_complexity(model)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

max_peak_allocated = 0
max_peak_reserved = 0

training_start = time.perf_counter()


# ==========================================
# TRAINING
# ==========================================

print("\n--- BẮT ĐẦU HUẤN LUYỆN ---")

try:

    for epoch in range(NUM_EPOCHS):

        epoch_start = time.perf_counter()

        # ----------------------------------
        # GPU MEMORY RESET
        # ----------------------------------

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()

        # ----------------------------------
        # TRAINING
        # ----------------------------------

        train_start = time.perf_counter()

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        train_time = time.perf_counter() - train_start

        # ----------------------------------
        # VALIDATION
        # ----------------------------------

        val_loss, val_acc = evaluate(
            model,
            val_loader,
            criterion,
            device
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        # ----------------------------------
        # GPU MEMORY
        # ----------------------------------

        if torch.cuda.is_available():

            memory_allocated = (
                torch.cuda.memory_allocated() / 1024**2
            )

            memory_reserved = (
                torch.cuda.memory_reserved() / 1024**2
            )

            peak_allocated = (
                torch.cuda.max_memory_allocated() / 1024**2
            )

            peak_reserved = (
                torch.cuda.max_memory_reserved() / 1024**2
            )

            max_peak_allocated = max(
                max_peak_allocated, peak_allocated
            )

            max_peak_reserved = max(
                max_peak_reserved, peak_reserved
            )

        else:
            memory_allocated = None
            memory_reserved = None
            peak_allocated = None
            peak_reserved = None

        # ----------------------------------
        # LEARNING RATE
        # ----------------------------------

        # LR actually used in this epoch
        current_lr = optimizer.param_groups[0]["lr"]

        # ----------------------------------
        # EPOCH TIME
        # ----------------------------------

        epoch_time = time.perf_counter() - epoch_start

        samples_per_sec = (
            len(train_loader.dataset) / train_time
            if train_time > 0 else 0
        )

        # ----------------------------------
        # UPDATE HISTORY
        # ----------------------------------

        current_run_history["train_loss"].append(
            round(train_loss, 4)
        )

        current_run_history["train_acc"].append(
            round(train_acc * 100, 2)
        )

        current_run_history["val_loss"].append(
            round(val_loss, 4)
        )

        current_run_history["val_acc"].append(
            round(val_acc * 100, 2)
        )

        # ----------------------------------
        # LOG EACH EPOCH TO JSON
        # ----------------------------------

        logger.log_epoch(epoch + 1, {

            "train_loss": round(train_loss, 4),
            "train_accuracy": round(train_acc * 100, 2),

            "val_loss": round(val_loss, 4),
            "val_accuracy": round(val_acc * 100, 2),

            "learning_rate": current_lr,

            "epoch_time_sec": round(epoch_time, 3),
            "samples_per_sec": round(samples_per_sec, 2),

            "memory_allocated_mb": (
                round(memory_allocated, 2)
                if memory_allocated is not None else None
            ),

            "memory_reserved_mb": (
                round(memory_reserved, 2)
                if memory_reserved is not None else None
            ),

            "peak_memory_allocated_mb": (
                round(peak_allocated, 2)
                if peak_allocated is not None else None
            ),

            "peak_memory_reserved_mb": (
                round(peak_reserved, 2)
                if peak_reserved is not None else None
            )
        })

        # ----------------------------------
        # PRINT PROGRESS
        # ----------------------------------

        print(
            f"Epoch [{epoch+1:02d}/{NUM_EPOCHS:02d}] | "
            f"Train Loss: {train_loss:.4f} - "
            f"Train Acc: {train_acc*100:.2f}% | "
            f"Val Loss: {val_loss:.4f} - "
            f"Val Acc: {val_acc*100:.2f}% | "
            f"LR: {current_lr:.6f} | "
            f"Time: {epoch_time:.2f}s"
        )

        # ----------------------------------
        # SAVE BEST CHECKPOINT
        # ----------------------------------

        if val_acc > best_val_acc:

            best_val_acc = val_acc

            torch.save(
                model.state_dict(),
                CHECKPOINT_PATH
            )

            logger.log_best_checkpoint(
                epoch + 1,
                round(val_acc * 100, 2)
            )

            print(
                f"  --> Đã lưu model tốt nhất: "
                f"Val Acc = {val_acc*100:.2f}%"
            )

        # ----------------------------------
        # UPDATE SCHEDULER
        # ----------------------------------

        scheduler.step()

except Exception as e:

    logger.log_event(
        "experiment_failed",
        {"error": str(e)}
    )

    logger.finish(status="failed")

    raise


# ==========================================
# SAVE TRAINING HISTORY
# ==========================================

training_time = time.perf_counter() - training_start

with open(HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(
        current_run_history,
        f,
        indent=4,
        ensure_ascii=False
    )


# ==========================================
# TESTING
# ==========================================

print("\n--- ĐÁNH GIÁ TRÊN TẬP TEST ---")

# Load best checkpoint
model.load_state_dict(
    torch.load(CHECKPOINT_PATH, map_location=device)
)

test_loss, test_acc = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print(
    f"Test Loss: {test_loss:.4f} | "
    f"Test Accuracy: {test_acc*100:.2f}%"
)

# Save test results
logger.log_test_results({
    "loss": round(test_loss, 4),
    "accuracy": round(test_acc * 100, 2)
})


# ==========================================
# FINISH EXPERIMENT
# ==========================================

logger.finish(
    status="completed",
    extra_metrics={
        "training_time_sec": round(training_time, 2),
        "best_val_accuracy": round(best_val_acc * 100, 2),
        "test_accuracy": round(test_acc * 100, 2),
        "max_peak_memory_allocated_mb": round(max_peak_allocated, 2),
        "max_peak_memory_reserved_mb": round(max_peak_reserved, 2)
    }
)


# ==========================================

print("\n--- EXPERIMENT COMPLETED ---")



--- BẮT ĐẦU HUẤN LUYỆN ---
Epoch [01/50] | Train Loss: 3.8093 - Train Acc: 11.95% | Val Loss: 3.3966 - Val Acc: 18.36% | LR: 0.000100 | Time: 54.14s
  --> Đã lưu model tốt nhất: Val Acc = 18.36%
Epoch [02/50] | Train Loss: 3.1719 - Train Acc: 22.22% | Val Loss: 2.8312 - Val Acc: 28.40% | LR: 0.000100 | Time: 56.45s
  --> Đã lưu model tốt nhất: Val Acc = 28.40%
Epoch [03/50] | Train Loss: 2.7832 - Train Acc: 29.57% | Val Loss: 2.6245 - Val Acc: 32.76% | LR: 0.000100 | Time: 56.43s
  --> Đã lưu model tốt nhất: Val Acc = 32.76%
Epoch [04/50] | Train Loss: 2.5003 - Train Acc: 35.25% | Val Loss: 2.3327 - Val Acc: 39.02% | LR: 0.000099 | Time: 56.39s
  --> Đã lưu model tốt nhất: Val Acc = 39.02%
Epoch [05/50] | Train Loss: 2.2781 - Train Acc: 39.74% | Val Loss: 2.2117 - Val Acc: 41.72% | LR: 0.000098 | Time: 53.18s
  --> Đã lưu model tốt nhất: Val Acc = 41.72%
Epoch [06/50] | Train Loss: 2.1005 - Train Acc: 44.02% | Val Loss: 2.0246 - Val Acc: 46.68% | LR: 0.000098 | Time: 54.67s
  --> Đã l

In [69]:
!python plot_experiment_results.py

Tìm thấy 1 experiment.

========== EXPERIMENT SUMMARY ==========

Experiment: ResNet18_Scratch_seed42
Best Epoch: 47
Best Val Accuracy: 66.96
Test Accuracy: 66.8
Training Time: 1846.45 sec
Peak GPU Memory: 824.83 MB

Đã hoàn thành vẽ biểu đồ.
Output directory: /home/devops/data/DL4CV/DL4CV/A1/Retrain/results/plots
